# 00 — Point preparation

Builds `01_Points/points_master.gpkg` from the EE survey points+polygons: assigns each point to its parent field (`poly_id`), densifies with synthetic points, drops co-located conflicts, assigns a stable `id`.

Runtime: ~10 min.

## Setup

In [ ]:
!pip -q install geemap earthengine-api geopandas shapely pyogrio rtree scipy

In [ ]:
import os, glob, shutil, json
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from scipy.spatial import cKDTree
import ee, geemap

from google.colab import drive
drive.mount('/content/drive')

## Project paths

```
Crop_Classification/
  01_Points/          labelled points, labels, CV folds
  02_RawTimeSeries/   per-acquisition extractions   S1/  S2/
  03_Features/        per-source feature tables     S1/ S2/ AlphaEarth/ Tessera/
  04_Master/          model-ready table + manifest
  05_QA/              diagnostics and provenance
  99_Deprecated/      superseded outputs, kept not deleted
```

In [ ]:
ROOT = '/content/drive/MyDrive/Crop_Classification'

DIR_POINTS   = f'{ROOT}/01_Points'
DIR_RAW_TS   = f'{ROOT}/02_RawTimeSeries'
DIR_FEATURES = f'{ROOT}/03_Features'
DIR_MASTER   = f'{ROOT}/04_Master'
DIR_QA       = f'{ROOT}/05_QA'
DIR_OLD      = f'{ROOT}/99_Deprecated'

SOURCES = ['S1', 'S2', 'AlphaEarth', 'Tessera']
for d in ([DIR_POINTS, DIR_RAW_TS, DIR_FEATURES, DIR_MASTER, DIR_QA, DIR_OLD] +
          [f'{DIR_RAW_TS}/{s}' for s in ['S1', 'S2']] +
          [f'{DIR_FEATURES}/{s}' for s in SOURCES]):
    os.makedirs(d, exist_ok=True)

POINTS_GPKG = f'{DIR_POINTS}/points_master.gpkg'
print("Folder tree ready.")

## Clean slate

Moves stale outputs to `99_Deprecated` (nothing deleted). Set `CONFIRM_CLEANUP = True` to apply.

In [ ]:
CONFIRM_CLEANUP = False

legacy = [p for p in
          glob.glob(f'{ROOT}/Data') + glob.glob(f'{ROOT}/Satellite_Data') +
          glob.glob(f'{ROOT}/02_RawTimeSeries/S1/*') + glob.glob(f'{ROOT}/03_Features/S1/*')
          if os.path.exists(p)]

if not legacy:
    print("Nothing legacy found.")
for p in legacy:
    n = len(glob.glob(p + '/**', recursive=True)) if os.path.isdir(p) else 1
    print(f"  {p.replace(ROOT + '/', ''):<55} {n:>6} item(s)")

if CONFIRM_CLEANUP:
    for p in legacy:
        dst = f"{DIR_OLD}/{os.path.basename(p)}"
        if not os.path.exists(dst):
            shutil.move(p, dst)
            print("moved ->", dst.replace(ROOT + '/', ''))
    print("Cleanup done.")
else:
    print("\nCONFIRM_CLEANUP is False — nothing moved.")

## Configuration

In [ ]:
EE_PROJECT   = 'ee-geographymanas'
EE_POINTS    = 'projects/ee-ranitsundarchatterjee/assets/Crop_Classification/Data_Points'
EE_POLYGONS  = 'projects/ee-ranitsundarchatterjee/assets/Crop_Classification/Data_polygons'

GRID_SPACING    = 15    # metres between candidate synthetic points
BOUNDARY_BUFFER = -5    # inward shrink; keeps synthetic points off field edges
MIN_DISTANCE    = 15    # minimum separation from any ORIGINAL point

ee.Authenticate()
ee.Initialize(project=EE_PROJECT)

## Load the survey data

In [ ]:
gdf_points   = geemap.ee_to_gdf(ee.FeatureCollection(EE_POINTS))
gdf_polygons = geemap.ee_to_gdf(ee.FeatureCollection(EE_POLYGONS))

print(f"Original points  : {len(gdf_points):,}")
print(f"Original polygons: {len(gdf_polygons):,}")
print("Point columns    :", list(gdf_points.columns))

utm_crs = gdf_polygons.estimate_utm_crs()
print("Working CRS      :", utm_crs)

pts_p  = gdf_points.to_crs(utm_crs)
poly_p = gdf_polygons.to_crs(utm_crs)

## Assign points to fields

`poly_id` is the CV group key — keeps a field's points on one side of any split.

In [ ]:
joined = gpd.sjoin(pts_p, poly_p[['geometry']], how='left', predicate='within')
joined = joined[~joined.index.duplicated(keep='first')]      # overlapping polygons

pts_p = pts_p.copy()
pts_p['poly_idx'] = joined['index_right'].values      # numeric, used for densification
pts_p['source']   = 'original'

n_orphan = int(pts_p['poly_idx'].isna().sum())
print(f"Inside a polygon : {len(pts_p) - n_orphan:,}")
print(f"No parent polygon: {n_orphan:,}  ({n_orphan/len(pts_p):.1%})")

# Field area, for the mixed-pixel discussion later
poly_area = poly_p.geometry.area
pts_p['field_area_m2'] = pts_p['poly_idx'].map(poly_area).values

print(f"\nField area (m²): median {poly_area.median():,.0f}, "
      f"10th pct {poly_area.quantile(0.10):,.0f}")
print(f"Fields smaller than one 10 m pixel (100 m²): {(poly_area < 100).mean():.1%}")
print(f"Fields smaller than 4 pixels (400 m²)      : {(poly_area < 400).mean():.1%}")

## Densification

Grid step == `MIN_DISTANCE`, so only candidate-to-original distance needs checking (one KD-tree per field). Don't set `GRID_SPACING` below `MIN_DISTANCE`.

In [ ]:
new_blocks = []

for poly_id, poly in poly_p.iterrows():
    existing = pts_p[pts_p['poly_idx'] == poly_id]
    if existing.empty:
        continue

    safe = poly.geometry.buffer(BOUNDARY_BUFFER)
    if safe.is_empty or not safe.is_valid:
        continue

    minx, miny, maxx, maxy = safe.bounds
    xs = np.arange(minx, maxx, GRID_SPACING)
    ys = np.arange(miny, maxy, GRID_SPACING)
    if xs.size == 0 or ys.size == 0:
        continue

    xx, yy = np.meshgrid(xs, ys)
    cand = np.column_stack([xx.ravel(), yy.ravel()])

    geoms = gpd.GeoSeries(gpd.points_from_xy(cand[:, 0], cand[:, 1]), crs=utm_crs)
    inside = geoms.within(safe).values
    if not inside.any():
        continue
    cand, geoms = cand[inside], geoms[inside]

    orig_xy = np.column_stack([existing.geometry.x.values, existing.geometry.y.values])
    dist, nearest = cKDTree(orig_xy).query(cand, k=1)
    keep = dist >= MIN_DISTANCE
    if not keep.any():
        continue

    block = existing.iloc[nearest[keep]].reset_index(drop=True)
    block['geometry'] = geoms[keep].reset_index(drop=True)
    block['source']   = 'synthetic'
    block['poly_idx'] = poly_id
    new_blocks.append(block)

gdf_new = (pd.concat(new_blocks, ignore_index=True) if new_blocks
           else gpd.GeoDataFrame(columns=pts_p.columns, crs=utm_crs))
print(f"Synthetic points generated: {len(gdf_new):,}")

## Combine and assign ids

Ids are assigned **after** densification, from rounded UTM coords — assigning before, or copying from a parent point, would collide keys silently.

In [ ]:
combined = gpd.GeoDataFrame(
    pd.concat([pts_p, gdf_new], ignore_index=True), geometry='geometry', crs=utm_crs)
print(f"Combined: {len(combined):,} rows")

# --- Co-located points -------------------------------------------------------
# The survey contains points recorded twice at the same location. Two points in the
# same place produce identical features, so one is redundant; and if their crop labels
# disagree, the ground truth contradicts itself and neither copy is usable.

coord = (combined.geometry.x.round(1).astype('int64').astype(str) + '_' +
         combined.geometry.y.round(1).astype('int64').astype(str))
combined['_coord'] = coord

grp = combined.groupby('_coord')
n_groups = int((grp.size() > 1).sum())
n_rows = int(grp.size()[grp.size() > 1].sum())
print(f"\nCo-located groups: {n_groups:,} covering {n_rows:,} rows")

if n_groups:
    conflict = grp['crop'].nunique()
    bad = conflict[conflict > 1].index
    print(f"  agreeing on crop  : {n_groups - len(bad):,} groups")
    print(f"  CONFLICTING crop  : {len(bad):,} groups  "
          f"({int(combined['_coord'].isin(bad).sum()):,} rows)")

    print("\n  by source:")
    print("   ", combined[combined['_coord'].duplicated(keep=False)]['source']
          .value_counts().to_dict())

    # Conflicting locations are ambiguous ground truth: drop every copy.
    # Agreeing locations are harmless duplicates: keep one.
    before = len(combined)
    combined = combined[~combined['_coord'].isin(bad)]
    combined = combined[~combined['_coord'].duplicated(keep='first')]
    print(f"\n  removed {before - len(combined):,} rows -> {len(combined):,} remain")

combined = combined.drop(columns='_coord').reset_index(drop=True)

# --- Group key ---------------------------------------------------------------
# A point with no parent polygon is not a bad point; it is its own field. Giving it a
# unique group id keeps it in the analysis instead of being dropped by the fold
# assignment in notebook 01, which would silently lose several percent of the dataset.
combined['poly_id'] = np.where(
    combined['poly_idx'].notna(),
    'F' + combined['poly_idx'].fillna(-1).astype('int64').astype(str),
    'O' + combined.index.astype(str))
combined = combined.drop(columns='poly_idx')

n_field = int(combined['poly_id'].str.startswith('F').sum())
n_orph  = int(combined['poly_id'].str.startswith('O').sum())
print(f"\nGroup key: {combined['poly_id'].nunique():,} groups "
      f"({n_field:,} points in mapped fields, {n_orph:,} standalone)")

# --- Unique id ---------------------------------------------------------------
combined['survey_id'] = combined['id'] if 'id' in combined.columns else np.nan
x = combined.geometry.x.round(1).astype('int64')
y = combined.geometry.y.round(1).astype('int64')
combined['id'] = 'p' + x.astype(str) + '_' + y.astype(str)

assert combined['id'].is_unique, (
    f"{combined['id'].duplicated().sum()} id collisions remain — "
    "the geometry de-duplication above did not catch everything")
print(f"IDs: {combined['id'].nunique():,} unique for {len(combined):,} rows")

## Clean up columns

In [ ]:
final = combined.to_crs(epsg=4326)
final = final.drop(columns=[c for c in final.columns if c.endswith(('_left', '_right'))],
                   errors='ignore')

final['longitude'] = final.geometry.x
final['latitude']  = final.geometry.y
if 'harvest' in final.columns:
    final['harvest'] = final['harvest'].replace('NA', np.nan)

lead = ['id', 'survey_id', 'poly_id', 'source', 'crop', 'district', 'state',
        'season', 'field_area_m2']
order = ([c for c in lead if c in final.columns] +
         [c for c in final.columns if c not in lead + ['geometry']] + ['geometry'])
final = final[order]
print(list(final.columns))

## Quality checks

In [ ]:
assert final['id'].is_unique, "duplicate ids"
assert final['poly_id'].notna().all(), "missing group key"
print("id unique              :", final['id'].is_unique)
print("every point has a group:", final['poly_id'].notna().all())

utm = final.to_crs(final.estimate_utm_crs())
key = utm.geometry.x.round(2).astype(str) + '_' + utm.geometry.y.round(2).astype(str)
print(f"distinct geometries    : {key.nunique():,} / {len(final):,}")
assert key.nunique() == len(final), "co-located points remain"

print(f"\nby source:\n{final['source'].value_counts().to_string()}")
print(f"\ngroups (fields)        : {final['poly_id'].nunique():,}")

ppf = final.groupby('poly_id').size()
print(f"points per group       : median {ppf.median():.0f}, mean {ppf.mean():.2f}, "
      f"max {ppf.max()}")
print(f"groups with 1 point    : {(ppf == 1).mean():.1%}")
print(f"points sharing a group : {(final.groupby('poly_id')['id'].transform('size') > 1).mean():.1%}")
print("\nThat last number is your pseudo-replication rate. It is the share of points")
print("that a random split could leak. Group-aware CV is still required, but the risk")
print("is proportional to this number, not to the raw point count.")

In [ ]:
support = pd.DataFrame({
    'points': final.groupby('crop').size(),
    'fields': final.groupby('crop')['poly_id'].nunique(),
})
support['points_per_field'] = (support['points'] / support['fields']).round(1)
print("Support by crop — 'fields' is your real sample size, not 'points':")
print(support.sort_values('fields').to_string())

In [ ]:
# The old data had betel_leaf and flower at exactly 2,562 points each. Check whether that
# was a genuine coincidence or a duplication artefact.
same = support['points'][support['points'].duplicated(keep=False)]
if len(same):
    sub = support.loc[same.index]
    print("Classes sharing an identical point count:")
    print(sub.to_string())
    if sub['fields'].duplicated(keep=False).all():
        print("\n*** FIELD counts match too — something is duplicated upstream. Investigate.")
    else:
        print("\nField counts differ, so this is coincidence, not duplication. No action.")
else:
    print("No two classes share an identical point count.")

## Save

In [ ]:
final.to_file(POINTS_GPKG, layer='points', driver='GPKG')
final.drop(columns='geometry').to_csv(f'{DIR_POINTS}/points_master.csv', index=False)
print("Saved:", POINTS_GPKG)

check = gpd.read_file(POINTS_GPKG, layer='points')
print(f"Verified: {len(check):,} rows, {check['id'].nunique():,} unique ids")

In [ ]:
with open(f'{DIR_QA}/00_points_qa.json', 'w') as f:
    json.dump({
        'generated': pd.Timestamp.now().isoformat(),
        'params': {'grid_spacing_m': GRID_SPACING, 'boundary_buffer_m': BOUNDARY_BUFFER,
                   'min_distance_m': MIN_DISTANCE, 'utm_crs': str(utm_crs)},
        'original_points': int(len(gdf_points)),
        'synthetic_points': int(len(gdf_new)),
        'total_points': int(len(final)),
        'unique_ids': int(final['id'].nunique()),
        'distinct_geometries': int(key.nunique()),
        'groups': int(final['poly_id'].nunique()),
        'points_in_mapped_fields': int(final['poly_id'].str.startswith('F').sum()),
        'standalone_points': int(final['poly_id'].str.startswith('O').sum()),
        'pseudo_replication_rate': float(
            (final.groupby('poly_id')['id'].transform('size') > 1).mean()),
        'support_by_crop': support.to_dict('index'),
    }, f, indent=2, default=str)
print("QA saved.")

---
### Next: `01_Labels_and_Folds.ipynb`

Densified points add no information, only field-size weight. Report accuracy with and without them.